In [2]:
from dotenv import load_dotenv
import os

from agents import Agent, Runner


In [3]:
load_dotenv(override=True)

True

In [4]:
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1" 

In [5]:
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1" 

In [6]:
first_agent = Agent(name="First Agent", instructions="You are a model to all geneegral questions", model="gpt-4o-mini")

In [7]:
first_runner = await Runner.run(first_agent, "Who is the current Nigeria President")

In [8]:
print(first_runner.final_output)

As of my last knowledge update in October 2023, the President of Nigeria is Bola Ahmed Tinubu. He assumed office on May 29, 2023. Please verify with up-to-date sources for the most recent information.


In [9]:
blog_writer = Agent(name="Blog Writer", instructions="You are to write a blog based off the topic I will send to you", model="gpt-4o-mini")

In [10]:
blog_runner = await Runner.run(blog_writer, "Write abt coding agent") 

In [11]:
blog_runner.final_output 

"### Unlocking the Future: The Rise of Coding Agents\n\nIn recent years, the conversation around artificial intelligence (AI) has shifted significantly toward automation, machine learning, and the revolutionary potential of coding agents. But what exactly is a coding agent, and how is it reshaping the landscape of software development and beyond? \n\n#### What is a Coding Agent?\n\nA coding agent is essentially an AI-powered tool or program that can assist in the coding process. These agents utilize advanced algorithms and machine learning models to write, debug, and optimize code, often learning from vast repositories of programming knowledge. They are designed to understand coding languages, frameworks, and best practices, enabling them to help developers complete tasks more efficiently.\n\n#### The Evolution of Coding Agents\n\nThe concept of automating coding tasks isn't new; however, recent advancements in natural language processing (NLP) and AI have catapulted coding agents into

In [12]:
from agents import Agent, Runner, function_tool 

In [13]:
@function_tool
def get_weather(city: str) -> str:
  """
    Get the current weather for a city

    Args:
    city: The name of the city to get the current weather
  """
  weather_data = {
    "london": "Cloudy, 15C",
    "tokyo": "Sunny, 22C",
    "new york": "Rainy, 18C"
  }
  return weather_data.get(city.lower(), f"Weather data not available for this {city}")

In [14]:
weather_agent = Agent(
    name="WeatherBot",
    instructions="You help users check the weather, use the get_weather tool when asked about the weather",
    model="gpt-4o-mini",
    tools=[get_weather]
) 

In [15]:
weather_runner = await Runner.run(weather_agent, "Whats the weather in Lagos")
print(weather_runner.final_output) 



I'm sorry, but it seems that the weather data is not available for Lagos at the moment. Please check back later or try a different location.


Sequential Tool Calling

In [16]:
@function_tool
def get_user(email: str)-> str:
    """ Look up the user ID from email"""
    return "user_123"

@function_tool
def get_user_orders(user_id: str)-> str:
    """ Get orders for User ID """
    return "Order 1: Laptop, Order 2: Mouse"

@function_tool
def get_order_status(user_id: str)-> str:
    """ Get status of an order """
    return "Order 1: Delivery in Progress, Order 2: Awaiting Payment Confirmantion"



order_agent = Agent(name="Order Assistant", 
instructions="Help users check their order status, First use their email to find UserID, then get the order and order status",
model="gpt-4o-mini",
tools=[get_user, get_user_orders, get_order_status]
)

runner = await Runner.run(order_agent, "What is the status of my order? My email is ayomidesowande@gmail.com")
print(runner.final_output)

Your order statuses are as follows:

- **Laptop**: Delivery in Progress
- **Mouse**: Awaiting Payment Confirmation


Parallel Tool Calling

In [17]:
@function_tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"{city}: Sunny, 25°C"

@function_tool
def get_news(topic: str) -> str:
    """Get latest news on a topic."""
    return f"Latest {topic} news: ..."

@function_tool
def get_stock(symbol: str) -> str:
    """Get stock price."""
    return f"{symbol}: $150.00"


agent = Agent(
    name="MorningBriefing",
    instructions="Provide a morning briefing with weather, news, and stock info.",
    tools=[get_weather, get_news, get_stock]
)

runner = await Runner.run(agent, "Give me my morning briefing for NYC, tech news, and AAPL")
print(runner.final_output)

Morning briefing:

- Weather (NYC): Sunny, 25°C
- Tech news: Latest tech news news: ...
- AAPL: $150.00


Conditional Tool Calling

In [18]:
@function_tool
def search_web(query: str) -> str:
    """Search the web for current information."""
    return f"Web results for '{query}': ..."

@function_tool
def search_database(query: str) -> str:
    """Search internal company database."""
    return f"Database results for '{query}': ..."

@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents."""
    return f"Document results for '{query}': ..."

agent = Agent(
    name="SmartSearch",
    instructions="""You are a search assistant.
    
    Choose the right search based on the query:
    - For current events/general info → use search_web
    - For company/employee info → use search_database
    - For policy/procedure questions → use search_documents
    """,
    model="gpt-4o-mini",
    tools=[search_web, search_database, search_documents]
)

runner = await Runner.run(agent, "Find the latest company policy on remote work.")

print(runner.final_output)

It seems that I couldn't find the specific latest company policy on remote work in the documents. Would you like to try a different search or check another type of information?


Multi-Agentic Systems


Building Multi-Agentic Systems with Handoff

In [19]:
from agents import Agent, Runner, handoff, function_tool

@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents."""
    return f"Document results for '{query}': ..."

billing_agent = Agent(
    name="BillingSpecialist",
    instructions="""You handle billing questions:
    - Payment issues
    - Refunds
    - Invoice requests
    Be helpful and resolve issues quickly.""",
    tools=[search_documents]
)

technical_agent = Agent(
    name="TechnicalSupport",
    instructions="""You handle technical issues:
    - Bug reports
    - How-to questions
    - Feature requests
    Ask clarifying questions to understand the issue."""
)

triage_agent = Agent(
    name="Triage",
    instructions="""You are the first point of contact. Route customers immediately:
- Billing/payment/charge issues → transfer to BillingSpecialist RIGHT AWAY
- Technical problems → transfer to TechnicalSupport RIGHT AWAY

Do NOT ask clarifying questions if the intent is already clear. Transfer immediately.""",
    model="gpt-4o-mini",
    handoffs=[
        handoff(billing_agent),
        handoff(technical_agent)
    ]
)


result = await Runner.run(triage_agent, "I want to know how to fix my machine")
print(result.final_output)

Sure — what kind of machine is it, and what’s wrong with it?

Please tell me:
- the machine type/model
- what it’s doing or not doing
- any error messages or sounds
- when the problem started
- what you’ve already tried

If you want, you can paste a photo of the error or describe the symptoms.



Building Multi-Agentic Systems with Agent-as-a-tool

In [20]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"


from agents import Agent, Runner, function_tool

# Create specialist agents
researcher = Agent(
    name="Researcher",
    instructions="You research topics thoroughly and return detailed findings.",
    model="gpt-4o"
)

writer = Agent(
    name="Writer", 
    instructions="You write clear, engaging content based on provided information.",
    model="gpt-4o"
)

@function_tool
async def research_topic(topic: str) -> str:
    """
    Research a topic thoroughly.
    
    Args:
        topic: The topic to research
    """
    result = await Runner.run(researcher, f"Research this topic: {topic}")
    return result.final_output

@function_tool
async def write_content(brief: str) -> str:
    """
    Write content based on a brief.
    
    Args:
        brief: The writing brief with topic and key points
    """
    result = await Runner.run(writer, brief)
    return result.final_output

orchestrator = Agent(
    name="ContentManager",
    instructions="""You manage content creation.
    
    When asked to create content:
    1. Use research_topic to gather information
    2. Use write_content to create the final piece
    3. Review and present the result
    """,
    model="gpt-4o-mini",
    tools=[research_topic, write_content]
)

runner = await Runner.run(
    orchestrator,
    "Create a blog post about the benefits of meditation"
)

print(runner.final_output)

Here's your blog post on the benefits of meditation:

---

**Unlocking the Power of Meditation: A Comprehensive Guide to Its Benefits**

In today’s fast-paced world, finding moments of calm can seem like an elusive dream. But what if the key to tranquility is just a breath away? Meditation, an ancient practice that dates back thousands of years, offers a practical solution to the relentless hustle of modern life. Whether you're seeking mental clarity or emotional resilience, meditation can be a transformative addition to your daily routine. Let’s dive into the myriad benefits of meditation and explore how you can start integrating it into your life today.

### Mental Health Benefits

Meditation is a powerful tool for enhancing mental well-being. Research indicates that regular meditation practice can significantly reduce symptoms of anxiety and depression. By encouraging mindfulness and fostering a non-judgmental awareness of the present moment, meditation helps break the cycle of nega